# Notebook 61 — Telemetry Dataset

**Telemetry datasets specify learned controller inputs**

Cluster telemetry becomes a learning substrate when traces are converted into time-indexed state, action, outcome, and constraint records.

This notebook starts the third arc of the repo: learning controllers from the second arc's adaptive serving system.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
import zipfile

FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 30,
    "axes.labelsize": 22,
    "legend.fontsize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
})

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    return path

def clean_axes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

def rounded_box(ax, xy, w, h, text, fontsize=15, lw=2):
    x, y = xy
    p = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.035,rounding_size=0.08",
                       linewidth=lw, edgecolor="black", facecolor="white")
    ax.add_patch(p)
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=fontsize)
    return p

def arrow(ax, start, end, lw=2, rad=0):
    a = FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=18, linewidth=lw,
                        color="black", connectionstyle=f"arc3,rad={rad}")
    ax.add_patch(a)
    return a


## Core equations

State vector:

\[
S(t)=\{q(t),u(t),m(t),\lambda(t),r(t),v(t)\}
\]

Telemetry dataset:

\[
D=\{S(t),\pi(t),S(t+1),y(t),c(t)\}_{t=1}^{T}
\]

Outcome label:

\[
y(t)=\Delta G(t)-\Delta L(t)-\Delta M(t)-\Delta S(t)-\Delta V(t)
\]

Learned controller target:

\[
\pi^*(t)=\arg\max_{\pi} \mathbb{E}[y(t+1)\mid S(t),\pi]
\]


In [ ]:
rng = np.random.default_rng(61)
T = 240
t = np.arange(T)
queue = 32 + 10*np.sin(t/19) + rng.normal(0, 3, T)
utilization = 0.68 + 0.16*np.sin(t/31 + 0.6) + rng.normal(0, 0.035, T)
memory = 0.56 + 0.17*np.sin(t/27 + 1.2) + rng.normal(0, 0.04, T)
arrival_rate = 0.50 + 0.22*np.sin(t/23 + 2.1) + rng.normal(0, 0.04, T)
reserve = np.clip(1.0 - utilization + rng.normal(0, 0.025, T), 0.05, 0.95)
verification = np.clip(0.30 + 0.34*(queue > 38) + 0.16*rng.random(T), 0.05, 0.95)
latency = 0.42 + 0.28*(queue/60) + 0.22*utilization + rng.normal(0, 0.025, T)

def choose_policy(q, u, m, r, lat):
    if q > 45 and r < 0.30: return "shed/shorten"
    if u > 0.78 and r > 0.35: return "scale-out"
    if q > 38 or m > 0.72: return "rebalance"
    if r > 0.55 and u < 0.68: return "reserve"
    return "steady"

policy = np.array([choose_policy(queue[i], utilization[i], memory[i], reserve[i], latency[i]) for i in range(T)])
gain = 0.42 + 0.30*(policy=="scale-out") + 0.18*(policy=="rebalance") + 0.10*(policy=="reserve") - 0.10*(policy=="shed/shorten")
latency_penalty = 0.18 + 0.38*np.clip(latency, 0, 1)
memory_penalty = 0.12 + 0.28*np.clip(memory, 0, 1)
spillover_penalty = 0.10 + 0.25*(policy=="scale-out")*(utilization > 0.78)
violation_penalty = 0.50*((latency > 0.82) | (memory > 0.82))
outcome = gain - latency_penalty - memory_penalty - spillover_penalty - violation_penalty + rng.normal(0, 0.03, T)

df = pd.DataFrame({
    "t": t,
    "queue_depth": np.clip(queue, 1, None),
    "gpu_utilization": np.clip(utilization, 0, 1),
    "memory_pressure": np.clip(memory, 0, 1),
    "arrival_rate": np.clip(arrival_rate, 0, 1),
    "reserve_capacity": reserve,
    "verification_load": verification,
    "latency": np.clip(latency, 0, 1),
    "policy": policy,
    "outcome_label": outcome,
})
df["constraint_violation"] = ((df["latency"] > 0.82) | (df["memory_pressure"] > 0.82)).astype(int)
df["next_queue_depth"] = df["queue_depth"].shift(-1)
df["next_gpu_utilization"] = df["gpu_utilization"].shift(-1)
df["next_memory_pressure"] = df["memory_pressure"].shift(-1)
df = df.dropna().reset_index(drop=True)
df.head()


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(16, 8)); clean_axes(ax)
ax.set_title("Telemetry datasets specify learned controller inputs", pad=28)
labels=["cluster\ntelemetry","trace\nschema","state\nwindows","action +\noutcome labels","training\ndataset","learned\ncontroller"]
x0,y,w,h,gap=.04,.54,.13,.16,.035
for i,label in enumerate(labels):
    x=x0+i*(w+gap); rounded_box(ax,(x,y),w,h,label,fontsize=15)
    if i<len(labels)-1: arrow(ax,(x+w,y+h/2),(x+w+gap*.82,y+h/2))
rounded_box(ax,(.20,.20),.60,.12,"Notebook 61 converts runtime traces into state, action, outcome, and constraint records.",fontsize=15)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_final_synthesis.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(16, 9)); clean_axes(ax)
ax.set_title("Notebook 61 starts the third systems arc", pad=25)
top=[("43","Resource\nAllocation"),("49","Adaptive\nInfrastructure"),("53","Distributed\nScheduling"),("59","Cluster\nOptimization")]
bottom=[("61","Telemetry\nDataset"),("67","Policy\nLearning"),("71","Offline\nEvaluation"),("73","Safety\nBounds"),("79","Adaptive\nController")]
for i,(n,label) in enumerate(top):
    x=.13+i*.19; rounded_box(ax,(x,.62),.14,.12,f"{n}\n{label}",fontsize=13)
    if i<len(top)-1: arrow(ax,(x+.14,.68),(x+.18,.68))
ax.text(.5,.52,"second arc: controller → infrastructure → distributed scheduling → cluster optimization",ha="center",fontsize=16)
ax.plot([.08,.92],[.47,.47],color="black",lw=2)
for i,(n,label) in enumerate(bottom):
    x=.06+i*.18; rounded_box(ax,(x,.26),.14,.12,f"{n}\n{label}",fontsize=13)
    if i<len(bottom)-1: arrow(ax,(x+.14,.32),(x+.18,.32))
arrow(ax,(.73,.62),(.13,.38),lw=2,rad=-.25)
ax.text(.5,.16,"third arc: telemetry dataset → policy learning → evaluation → safety → controller",ha="center",fontsize=16)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_repository_transition.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(15,8)); clean_axes(ax)
ax.set_title("Trace schema records the serving loop", pad=25)
cols=["time","request","queue","util","memory","policy","latency","outcome","constraint"]
xs=np.linspace(.05,.90,len(cols))
for x,c in zip(xs,cols): rounded_box(ax,(x,.58),.085,.12,c,fontsize=13)
for i in range(len(cols)-1): arrow(ax,(xs[i]+.085,.64),(xs[i+1],.64))
rounded_box(ax,(.18,.28),.60,.13,"Each row aligns observed state, selected policy, measured outcome, and violated constraints.",fontsize=14)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_trace_schema.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(15,8)); clean_axes(ax)
ax.set_title("Event streams become time-indexed tables", pad=25)
for i,txt in enumerate(["queue update","GPU sample","memory sample","policy decision","latency result","constraint flag"]):
    y=.78-i*.10; rounded_box(ax,(.08,y),.22,.07,txt,fontsize=12); arrow(ax,(.30,y+.035),(.44,.52),lw=1.6)
x0,y0=.48,.30; headers=["t","S(t)","π(t)","S(t+1)","y(t)","c(t)"]
for j,htxt in enumerate(headers):
    ax.add_patch(Rectangle((x0+j*.075,y0+.30),.075,.08,fill=False,lw=1.5)); ax.text(x0+j*.075+.0375,y0+.34,htxt,ha="center",va="center",fontsize=12)
for row in range(4):
    for j in range(len(headers)):
        ax.add_patch(Rectangle((x0+j*.075,y0+.22-row*.07),.075,.07,fill=False,lw=1)); ax.text(x0+j*.075+.0375,y0+.255-row*.07,"·",ha="center",va="center",fontsize=16)
arrow(ax,(.36,.52),(.46,.52),lw=2.4); ax.text(.5,.18,"Raw logs become aligned samples for controller training.",ha="center",fontsize=16)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_event_stream_to_table.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(14,8)); clean_axes(ax)
ax.set_title("State vector maps live telemetry into controller inputs", pad=25)
rounded_box(ax,(.39,.46),.22,.13,"State vector\nS(t)",fontsize=18)
items=[("q(t)\nqueue",.10,.70),("u(t)\nutilization",.39,.78),("m(t)\nmemory",.69,.70),("λ(t)\narrival",.10,.25),("r(t)\nreserve",.39,.14),("v(t)\nverification",.69,.25)]
for label,x,y in items:
    rounded_box(ax,(x,y),.18,.10,label,fontsize=14); arrow(ax,(x+.09,y+.05),(.50,.525),lw=1.8)
ax.text(.5,.36,r"$S(t)=\{q(t),u(t),m(t),\lambda(t),r(t),v(t)\}$",ha="center",fontsize=22)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_state_vector_map.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(14,8)); clean_axes(ax)
ax.set_title("Outcome labels combine gain and constraint costs", pad=25)
labels=[("throughput\nchange ΔG",.08,.62),("latency\nchange ΔL",.27,.62),("memory\nchange ΔM",.46,.62),("spillover\nchange ΔS",.65,.62),("violation\nchange ΔV",.84,.62)]
for text,x,y in labels: rounded_box(ax,(x,y),.13,.12,text,fontsize=12)
rounded_box(ax,(.32,.30),.36,.12,"outcome label\ny(t)",fontsize=18)
for text,x,y in labels: arrow(ax,(x+.065,y),(.50,.42),lw=1.8)
ax.text(.5,.18,r"$y(t)=\Delta G(t)-\Delta L(t)-\Delta M(t)-\Delta S(t)-\Delta V(t)$",ha="center",fontsize=22)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_label_construction.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(15,7))
ax.set_title("Windowed telemetry gives the controller temporal context", pad=22)
window=16; series=df["queue_depth"].to_numpy()[:90]
ax.plot(np.arange(len(series)),series,marker="o",linewidth=2,label="queue depth")
for start in [8,26,44,62]:
    ax.axvspan(start,start+window,alpha=.12); ax.text(start+window/2,max(series)+2,f"window\n{start}:{start+window}",ha="center",fontsize=11)
ax.set_xlabel("time step"); ax.set_ylabel("queue depth"); ax.grid(True,alpha=.3); ax.legend(); savefig("61_windowed_dataset.png")


In [ ]:
# Figure
regime=[]
for _,row in df.iterrows():
    if row["queue_depth"]>45 and row["reserve_capacity"]<.30: regime.append("shed/shorten")
    elif row["gpu_utilization"]>.78 and row["reserve_capacity"]>.35: regime.append("scale-out")
    elif row["queue_depth"]>38 or row["memory_pressure"]>.72: regime.append("rebalance")
    elif row["reserve_capacity"]>.55 and row["gpu_utilization"]<.68: regime.append("reserve")
    else: regime.append("steady")
df["regime"]=regime
fig, ax = plt.subplots(figsize=(12,7)); ax.set_title("Dataset coverage must balance operating regimes", pad=20)
counts=df["regime"].value_counts().reindex(["steady","reserve","rebalance","scale-out","shed/shorten"]).fillna(0)
ax.bar(counts.index,counts.values); ax.set_ylabel("samples"); ax.set_xlabel("operating regime"); ax.grid(axis="y",alpha=.3); plt.xticks(rotation=20,ha="right"); savefig("61_regime_balance.png")


In [ ]:
# Figure
x=np.linspace(.05,.95,80); y=np.linspace(.05,.95,80); X,Y=np.meshgrid(x,y)
Z=.45+.45*np.exp(-((X-.62)**2/.08+(Y-.62)**2/.10))-.45*(Y>.82)*(X<.35)-.25*(Y<.30)*(X>.75)-.18*(X<.18); Z=np.clip(Z,-.25,1)
fig, ax = plt.subplots(figsize=(10,8)); ax.set_title("Policy outcome surface", pad=20)
cs=ax.contourf(X,Y,Z,levels=16); cb=fig.colorbar(cs,ax=ax); cb.set_label("expected outcome label")
sample=df.sample(36,random_state=61); ax.scatter(sample["reserve_capacity"],sample["gpu_utilization"],s=45,edgecolor="black")
ax.set_xlabel("reserve capacity"); ax.set_ylabel("active load"); ax.text(.57,.66,"high-value\ntraining region",fontsize=14); ax.text(.16,.86,"constraint\nrisk",fontsize=14); savefig("61_policy_outcome_surface.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(15,6)); ax.set_title("Time-aware split avoids leakage", pad=20)
n=len(df); train_end=int(.65*n); val_end=int(.82*n)
ax.axvspan(0,train_end,alpha=.25,label="train"); ax.axvspan(train_end,val_end,alpha=.25,label="validation"); ax.axvspan(val_end,n,alpha=.25,label="test")
ax.plot(df.index,df["queue_depth"],linewidth=1.8,label="queue depth"); ax.axvline(train_end,color="black",linestyle="--"); ax.axvline(val_end,color="black",linestyle="--")
ax.set_xlabel("time index"); ax.set_ylabel("queue depth"); ax.grid(True,alpha=.25); ax.legend(loc="upper right"); savefig("61_train_eval_split.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(14,8)); clean_axes(ax)
ax.set_title("Quality checks protect learned controller inputs", pad=25)
checks=[("missing\ntelemetry",.08,.62),("delayed\nlabels",.31,.62),("outlier\nspikes",.54,.62),("policy\nimbalance",.77,.62),("clock\nskew",.20,.34),("duplicate\nrequests",.43,.34),("constraint\ncoverage",.66,.34)]
for text,x,y in checks: rounded_box(ax,(x,y),.15,.10,text,fontsize=13)
rounded_box(ax,(.25,.12),.50,.10,"validated dataset D",fontsize=18)
for text,x,y in checks: arrow(ax,(x+.075,y),(.50,.22),lw=1.5)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_quality_checks.png")


In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(16,8)); clean_axes(ax)
ax.set_title("Telemetry dataset opens the learning-controller arc", pad=25)
arc=[("61","Telemetry\nDataset"),("67","Policy\nLearning"),("71","Offline\nEvaluation"),("73","Safety\nBounds"),("79","Adaptive\nController")]
for i,(n,label) in enumerate(arc):
    x=.07+i*.18; rounded_box(ax,(x,.54),.14,.13,f"{n}\n{label}",fontsize=13)
    if i<len(arc)-1: arrow(ax,(x+.14,.605),(x+.18,.605))
rounded_box(ax,(.22,.25),.56,.12,"Third arc: use telemetry to learn policies before allowing adaptive runtime control.",fontsize=15)
ax.set_xlim(0,1); ax.set_ylim(0,1); savefig("61_next_arc_map.png")


## Dataset export

The next cell writes a compact CSV artifact and a zip of all generated figures.

In [ ]:
csv_path = DATA_DIR / "61_telemetry_dataset.csv"
df.to_csv(csv_path, index=False)
zip_path = Path("61_telemetry_dataset_figures.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname=str(csv_path))
    for path in sorted(FIG_DIR.glob("61_*.png")):
        zf.write(path, arcname=str(path))
print(f"wrote {csv_path}")
print(f"wrote {zip_path}")


## Download cell

Run this in Colab to download the notebook outputs.

In [ ]:
# Colab download cell — keep this at the end
try:
    from google.colab import files
    files.download("61_telemetry_dataset_figures.zip")
except Exception as exc:
    print("Download helper is available in Google Colab.")
    print("Local file:", "61_telemetry_dataset_figures.zip")
    print("Details:", exc)
